# MtbAIM

Prediction of *Mycobacterium tuberculosis* activity and MIC-family endpoints from SMILES.

**Input options**
- Paste SMILES directly. Separate multiple entries with semicolons (`;`).
- Upload a `.txt` file with one SMILES per line.
- Upload a `.csv` file containing a SMILES column.

**Maximum:** 100 compounds per run.


In [ ]:
#@title Predict { display-mode: "form", form-width: "100%" }
#@markdown Choose how you want to provide compounds.
input_method = "Paste SMILES"  #@param ["Paste SMILES", "Upload .txt or .csv"]
#@markdown If **Paste SMILES** is selected, enter one SMILES or separate multiple SMILES with semicolons (`;`).
smiles_input = ""  #@param {type:"string"}
#@markdown Automatically download the complete CSV when prediction finishes.
download_results = True  #@param {type:"boolean"}

import os
import sys
import re
import json
import stat
import shutil
import zipfile
import subprocess
import importlib.util
import uuid
import time
from pathlib import Path

HF_REPO_ID = "jasdeep002/MtbAIM"
HF_FILENAME = "MTB_AI_predictor_v1.zip"
HF_REPO_TYPE = "dataset"
HF_REVISION = "main"

SAFE_CWD = Path("/content")
WORK = SAFE_CWD / "MtbAIM"
BUNDLE_DIR = WORK / "bundle"
OUTPUT_DIR = WORK / "results"
MAX_ROWS = 100

def _run(cmd, cwd=None):
    run_cwd = Path(cwd).resolve() if cwd is not None else SAFE_CWD.resolve()
    if not run_cwd.exists():
        run_cwd.mkdir(parents=True, exist_ok=True)
    p = subprocess.run(
        cmd,
        cwd=str(run_cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if p.returncode != 0:
        tail = (p.stderr or p.stdout or "")[-2500:]
        raise RuntimeError("A required setup command failed.\n" + tail)
    return p

def _safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError("The model archive failed its integrity check.")
        infos = zf.infolist()
        if len(infos) > 200:
            raise RuntimeError("Unexpected model archive structure.")
        total = sum(int(x.file_size) for x in infos)
        if total > 4 * 1024**3:
            raise RuntimeError("The model archive exceeds the allowed extraction size.")
        for info in infos:
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError("The model archive contains an unsupported symbolic link.")
            target = (destination / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError("Unsafe path detected in the model archive.")
        zf.extractall(destination)

def _parse_pasted_text(text):
    text = str(text or "").strip()
    if not text:
        raise ValueError("Enter at least one SMILES.")
    parts = []
    for line in text.replace("\r", "\n").split("\n"):
        for item in line.split(";"):
            item = item.strip()
            if item:
                parts.append(item)
    if not parts:
        raise ValueError("No SMILES were found.")
    return parts

def _find_smiles_column(df):
    preferred = ("smiles", "canonical_smiles", "standardized_smiles")
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for key in preferred:
        if key in lower_map:
            return lower_map[key]
    raise ValueError(
        "CSV must contain a SMILES column, for example: "
        "SMILES, smiles, canonical_smiles, or standardized_smiles."
    )

def _read_input_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".txt":
        lines = [
            x.strip()
            for x in path.read_text(encoding="utf-8-sig").splitlines()
            if x.strip()
        ]
        if not lines:
            raise ValueError("The uploaded text file contains no SMILES.")
        import pandas as pd
        return pd.DataFrame({
            "Compound_ID": [f"Compound_{i+1:03d}" for i in range(len(lines))],
            "SMILES": lines,
        }), "SMILES"
    if suffix == ".csv":
        import pandas as pd
        df = pd.read_csv(path)
        if len(df) == 0:
            raise ValueError("The uploaded CSV contains no rows.")
        return df.reset_index(drop=True), _find_smiles_column(df)
    raise ValueError("Upload a .txt or .csv file.")

def _enforce_limit(n):
    n = int(n)
    if n < 1:
        raise ValueError("No compounds were supplied.")
    if n > MAX_ROWS:
        raise ValueError(
            f"A maximum of {MAX_ROWS} compounds can be submitted per run. "
            f"Received {n}."
        )

def _prepare_bundle():
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

    # A previous notebook run leaves the process inside the extracted bundle.
    # Move to a stable directory before refreshing /content/MtbAIM; otherwise
    # deleting WORK can delete the process current working directory and make
    # pip/os.getcwd() fail on rerun.
    SAFE_CWD.mkdir(parents=True, exist_ok=True)
    os.chdir(SAFE_CWD)

    WORK.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    _run([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--disable-pip-version-check",
        "huggingface_hub>=0.24",
    ])

    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    info = api.dataset_info(repo_id=HF_REPO_ID, revision=HF_REVISION)
    commit = getattr(info, "sha", None)
    if not commit:
        raise RuntimeError("Could not resolve the published model version.")

    available = {
        getattr(x, "rfilename", None)
        for x in getattr(info, "siblings", [])
    }
    if HF_FILENAME not in available:
        raise FileNotFoundError("The published model archive was not found.")

    archive = Path(hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        repo_type=HF_REPO_TYPE,
        revision=commit,
    ))
    if not archive.exists() or archive.stat().st_size == 0:
        raise RuntimeError("The downloaded model archive is empty.")

    if BUNDLE_DIR.exists():
        shutil.rmtree(BUNDLE_DIR)
    _safe_extract(archive, BUNDLE_DIR)

    configs = list(BUNDLE_DIR.rglob("deployment_config.json"))
    if len(configs) != 1:
        raise RuntimeError("The model archive has an unexpected structure.")
    root = configs[0].parent.resolve()

    validator = root / "validate_bundle.py"
    if not validator.exists():
        raise RuntimeError("Model validation information is missing.")
    check = _run([sys.executable, str(validator)], cwd=root)
    if "BUNDLE VALIDATION PASS" not in check.stdout:
        raise RuntimeError("Model validation did not pass.")

    req = root / "requirements_colab.txt"
    if not req.exists():
        raise RuntimeError("Runtime requirements are missing.")
    _run([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--disable-pip-version-check",
        "-r", str(req),
    ])

    check2 = _run([sys.executable, str(validator)], cwd=root)
    if "BUNDLE VALIDATION PASS" not in check2.stdout:
        raise RuntimeError("Model validation did not pass after setup.")

    return root, commit

def _load_predictor(root):
    os.chdir(root)
    cfg = json.loads((root / "deployment_config.json").read_text())
    if int(cfg.get("max_smiles_per_run", -1)) != MAX_ROWS:
        raise RuntimeError("Published batch-size configuration is inconsistent.")

    engine_path = root / cfg["inference_engine"]
    spec = importlib.util.spec_from_file_location("mtbaim_engine", engine_path)
    engine = importlib.util.module_from_spec(spec)
    sys.modules["mtbaim_engine"] = engine
    spec.loader.exec_module(engine)

    try:
        from rdkit import RDLogger
        RDLogger.DisableLog("rdApp.error")
        RDLogger.DisableLog("rdApp.warning")
    except Exception:
        pass

    bundles, features, _manifest = engine.load_and_validate_step29()
    if len(features) != 2265 or len(bundles) != 8:
        raise RuntimeError("Published predictor configuration is incomplete.")
    return cfg, engine, bundles, features

def _validate_runtime(root, cfg, engine, bundles, feature_columns):
    import numpy as np
    import pandas as pd

    parsed = engine.parse_feature_contract(feature_columns)
    rd_pos = np.asarray([x[0] for x in parsed[0]], dtype=int)
    mg_pos = np.asarray([x[0] for x in parsed[1]], dtype=int)

    qc = pd.read_csv(root / cfg["compact_feature_qc"], compression="infer")
    required_meta = ["dataset", "source_row_index", "row_id", "canonical_smiles"]
    if list(qc.columns[:4]) != required_meta:
        raise RuntimeError("Feature validation reference is malformed.")
    if list(map(str, qc.columns[4:])) != list(map(str, feature_columns)):
        raise RuntimeError("Feature validation schema mismatch.")

    for dataset in ("DS", "DR"):
        block = qc[qc["dataset"].astype(str) == dataset].reset_index(drop=True)
        if len(block) == 0:
            raise RuntimeError("Feature validation reference is incomplete.")

        x_new = np.empty((len(block), len(feature_columns)), dtype=np.float32)
        for i, smi in enumerate(block["canonical_smiles"].astype(str)):
            mol, _canon, _status = engine.canonicalize_smiles(smi)
            if mol is None:
                raise RuntimeError("A validation structure could not be parsed.")
            x_new[i], _ = engine.molecule_feature_vector(
                mol, feature_columns, parsed
            )

        raw = block[feature_columns].apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=np.float64)
        x_old, _ = engine.sanitize_numeric_matrix(raw)

        if not np.array_equal(
            x_new[:, mg_pos], x_old[:, mg_pos], equal_nan=True
        ):
            raise RuntimeError("Fingerprint reproduction validation failed.")
        if not np.array_equal(
            np.isnan(x_new[:, rd_pos]), np.isnan(x_old[:, rd_pos])
        ):
            raise RuntimeError("Descriptor missing-value validation failed.")
        if not np.allclose(
            x_new[:, rd_pos], x_old[:, rd_pos],
            rtol=1e-6, atol=1e-7, equal_nan=True
        ):
            raise RuntimeError("Descriptor reproduction validation failed.")

    smoke = pd.read_csv(root / cfg["smoke_prediction_qc"], compression="infer")
    required = {"task", "canonical_smiles", "expected_step29_prediction"}
    if not required.issubset(smoke.columns):
        raise RuntimeError("Prediction validation reference is malformed.")

    cache = {}
    actual = []
    labels = []
    for row in smoke.itertuples(index=False):
        task = str(row.task)
        smi = str(row.canonical_smiles)
        if smi not in cache:
            mol, _canon, _status = engine.canonicalize_smiles(smi)
            if mol is None:
                raise RuntimeError("A prediction-validation structure is invalid.")
            vec, _ = engine.molecule_feature_vector(
                mol, feature_columns, parsed
            )
            cache[smi] = vec

        b = bundles[task]
        xt = b["imputer"].transform(
            cache[smi].reshape(1, -1)
        ).astype(np.float32, copy=False)

        if task.endswith("_CLASSIFICATION"):
            classes = list(b["model"].classes_)
            pos = b.get("positive_class", 1)
            pred = float(
                b["model"].predict_proba(xt)[0, classes.index(pos)]
            )
            labels.append(
                "Active" if pred >= float(b["threshold"]) else "Non-active"
            )
        else:
            pred = float(np.asarray(b["model"].predict(xt), dtype=float)[0])
            labels.append("")
        actual.append(pred)

    expected = pd.to_numeric(
        smoke["expected_step29_prediction"], errors="raise"
    ).to_numpy(dtype=float)
    if not np.allclose(
        np.asarray(actual, dtype=float),
        expected,
        rtol=1e-6,
        atol=1e-7,
    ):
        raise RuntimeError("Reference prediction validation failed.")

    if "derived_label" in smoke.columns:
        for i, row in smoke.iterrows():
            if str(row["task"]).endswith("_CLASSIFICATION"):
                expected_label = str(row["derived_label"])
                if expected_label and expected_label.lower() != "nan":
                    if labels[i] != expected_label:
                        raise RuntimeError(
                            "Reference classification validation failed."
                        )

def _predict(df, smiles_col, root, cfg, engine, bundles, feature_columns):
    import numpy as np
    import pandas as pd

    x, canonical, status, query_fps = engine.batch_features(
        df[smiles_col].tolist(), feature_columns
    )
    valid = np.asarray(
        [str(s).startswith("OK") for s in status],
        dtype=bool,
    )

    additions = {
        "MtbAIM_canonical_smiles": canonical,
        "MtbAIM_smiles_status": status,
    }
    additions.update(engine.predict_matrix(x, valid, bundles))

    training_fps = engine.build_training_fps_by_task()
    task_names = (
        "DS_CLASSIFICATION",
        "DR_CLASSIFICATION",
        "DS_MIC_REGRESSION",
        "DS_MIC90_REGRESSION",
        "DS_MIC99_REGRESSION",
        "DR_MIC_REGRESSION",
        "DR_MIC90_REGRESSION",
        "DR_MIC99_REGRESSION",
    )
    for task in task_names:
        if task not in training_fps:
            raise RuntimeError("Applicability reference is incomplete.")
        sims = np.full(len(df), np.nan, dtype=float)
        for i, bits in enumerate(query_fps):
            if bits is not None:
                sims[i] = engine.nearest_training_similarity(
                    bits, training_fps[task]
                )
        additions[f"{task}_max_train_tanimoto"] = sims

    result = pd.concat(
        [df.reset_index(drop=True), pd.DataFrame(additions)],
        axis=1,
        copy=False,
    )

    stamp = time.strftime("%Y%m%d_%H%M%S")
    out_file = OUTPUT_DIR / f"MtbAIM_predictions_{stamp}_{uuid.uuid4().hex[:6]}.csv"
    result.to_csv(out_file, index=False)

    summary = pd.DataFrame({
        "Compound": (
            df["Compound_ID"].astype(str)
            if "Compound_ID" in df.columns
            else [f"Compound_{i+1:03d}" for i in range(len(df))]
        ),
        "SMILES": df[smiles_col].astype(str),
        "Status": result["MtbAIM_smiles_status"],
        "DS activity": result.get("DS_activity_call"),
        "DS P(active)": result.get("DS_probability_active"),
        "DR activity": result.get("DR_activity_call"),
        "DR P(active)": result.get("DR_probability_active"),
        "DS MIC (µM)": result.get("DS_MIC_uM"),
        "DS MIC90 (µM)": result.get("DS_MIC90_uM"),
        "DS MIC99 (µM)": result.get("DS_MIC99_uM"),
        "DR MIC (µM)": result.get("DR_MIC_uM"),
        "DR MIC90 (µM)": result.get("DR_MIC90_uM"),
        "DR MIC99 (µM)": result.get("DR_MIC99_uM"),
        "DS similarity": result.get(
            "DS_CLASSIFICATION_max_train_tanimoto"
        ),
        "DR similarity": result.get(
            "DR_CLASSIFICATION_max_train_tanimoto"
        ),
    })

    for col in (
        "DS P(active)", "DR P(active)", "DS similarity", "DR similarity"
    ):
        summary[col] = pd.to_numeric(
            summary[col], errors="coerce"
        ).round(4)

    for col in (
        "DS MIC (µM)", "DS MIC90 (µM)", "DS MIC99 (µM)",
        "DR MIC (µM)", "DR MIC90 (µM)", "DR MIC99 (µM)",
    ):
        summary[col] = pd.to_numeric(
            summary[col], errors="coerce"
        ).map(lambda v: float(f"{v:.4g}") if pd.notna(v) else float("nan"))

    return result, summary, out_file

print("Preparing predictor...")

if input_method == "Paste SMILES":
    smiles = _parse_pasted_text(smiles_input)
    _enforce_limit(len(smiles))
    import pandas as pd
    input_df = pd.DataFrame({
        "Compound_ID": [f"Compound_{i+1:03d}" for i in range(len(smiles))],
        "SMILES": smiles,
    })
    smiles_col = "SMILES"
else:
    from google.colab import files
    print("Upload one .txt or .csv file.")
    uploaded = files.upload()
    accepted = [
        Path(name)
        for name in uploaded
        if Path(name).suffix.lower() in (".txt", ".csv")
    ]
    if len(accepted) != 1 or len(uploaded) != 1:
        raise ValueError("Upload exactly one .txt or .csv file.")
    input_df, smiles_col = _read_input_file(accepted[0])
    _enforce_limit(len(input_df))

root, _published_revision = _prepare_bundle()
cfg, engine, bundles, feature_columns = _load_predictor(root)
_validate_runtime(root, cfg, engine, bundles, feature_columns)

print(f"Predicting {len(input_df)} compound(s)...")
full_result, summary, result_file = _predict(
    input_df, smiles_col, root, cfg, engine, bundles, feature_columns
)

from IPython.display import display
display(summary)

n_valid = int(
    full_result["MtbAIM_smiles_status"].astype(str).str.startswith("OK").sum()
)
n_invalid = int(len(full_result) - n_valid)

print(
    f"\nPrediction complete: {len(full_result)} compound(s) | "
    f"{n_valid} valid | {n_invalid} invalid"
)
print("Results:", result_file.name)

if download_results:
    from google.colab import files
    files.download(str(result_file))

### Output

The table reports DS and DR activity predictions together with MIC, MIC90, and MIC99 estimates.

The downloaded CSV contains the complete prediction output, including pMIC, concentration values, interval estimates, canonical SMILES, and nearest-training-set similarity.
